# Mitchell’s OmniMart Retail Data Platform
## Silver Layer - Data Transformation & Standardization

### Project Overview
This notebook transforms raw Bronze layer retail datasets into curated Silver layer datasets within the Microsoft Fabric Lakehouse architecture.

The Silver layer focuses on:
- Data cleansing
- Standardization
- Validation
- Deduplication
- Business rule enforcement

The transformed datasets will serve as the foundation for Gold layer dimensional modeling and enterprise reporting.


## Technologies Used

- Microsoft Fabric
- OneLake Lakehouse
- PySpark
- Delta Tables
- Medallion Architecture
- Data Validation & Transformation

## Silver Layer Objectives

The Silver layer is responsible for transforming raw Bronze datasets into validated and standardized datasets.

Transformation goals include:
- Removing duplicates
- Standardizing text formats
- Enforcing business rules
- Validating transactional relationships
- Improving data quality
- Preparing data for dimensional modeling

## 1. Load Bronze Layer Tables

This section loads all Bronze layer Delta tables into Spark DataFrames for transformation and cleansing.

Source tables:
- bronze_customers
- bronze_products
- bronze_orders
- bronze_order_items
- bronze_returns
- bronze_inventory

In [30]:
df_bronze_customers = spark.table("bronze_customers")
df_bronze_products = spark.table("bronze_products")
df_bronze_orders = spark.table("bronze_orders")
df_bronze_order_items = spark.table("bronze_order_items")
df_bronze_returns = spark.table("bronze_returns")
df_bronze_inventory = spark.table("bronze_inventory")
df_bronze_locations = spark.table("bronze_locations")

StatementMeta(, 6456375b-7f01-4a0d-b876-a18e7f14e1f4, 31, Finished, Available, Finished, False)

## 2. Customer Data Transformation

This section cleans and standardizes customer data.

Transformation logic:
- Remove duplicate customer records
- Standardize customer names
- Normalize email formatting
- Standardize city and regional naming conventions

Output:
- silver_customers

In [31]:
from pyspark.sql.functions import *

df_silver_customers = (
    df_bronze_customers
    .dropDuplicates(["customer_id"])
    .withColumn("customer_name", initcap(trim(col("customer_name"))))
    .withColumn("email", lower(trim(col("email"))))
    .withColumn("city", initcap(trim(col("city"))))
)

StatementMeta(, 6456375b-7f01-4a0d-b876-a18e7f14e1f4, 32, Finished, Available, Finished, False)

In [32]:
df_silver_customers.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_customers")

StatementMeta(, 6456375b-7f01-4a0d-b876-a18e7f14e1f4, 33, Finished, Available, Finished, False)

## 3. Product Data Transformation

This section validates and standardizes retail product data.

Transformation logic:
- Validate positive pricing values
- Ensure selling price exceeds cost price
- Standardize category and subcategory formatting
- Normalize brand names

Output:
- silver_products

In [33]:
df_silver_products = (
    df_bronze_products
    .filter(col("selling_price") > 0)
    .filter(col("cost_price") > 0)
    .filter(col("selling_price") >= col("cost_price"))
    .withColumn("category", initcap(trim(col("category"))))
)

StatementMeta(, 6456375b-7f01-4a0d-b876-a18e7f14e1f4, 34, Finished, Available, Finished, False)

In [34]:
df_silver_products.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_products")

StatementMeta(, 6456375b-7f01-4a0d-b876-a18e7f14e1f4, 35, Finished, Available, Finished, False)

## 4. Orders Data Transformation

This section transforms customer order transaction data.

Transformation logic:
- Validate order statuses
- Standardize sales channel formatting
- Normalize payment method values
- Prepare partitioned transaction data

Partition Strategy:
- order_year
- order_month

Output:
- silver_orders

In [35]:
valid_statuses = ["Completed", "Returned", "Cancelled"]

df_silver_orders = (
    df_bronze_orders
    .filter(col("order_status").isin(valid_statuses))
    .withColumn("sales_channel", initcap(trim(col("sales_channel"))))
    .withColumn("payment_method", initcap(trim(col("payment_method"))))
)

StatementMeta(, 6456375b-7f01-4a0d-b876-a18e7f14e1f4, 36, Finished, Available, Finished, False)

In [36]:
df_silver_orders.write \
    .mode("overwrite") \
    .partitionBy("order_year", "order_month") \
    .format("delta") \
    .saveAsTable("silver_orders")

StatementMeta(, 6456375b-7f01-4a0d-b876-a18e7f14e1f4, 37, Finished, Available, Finished, False)

## 5. Order Items Fact Transformation

This section transforms line-level transactional sales data.

Transformation logic:
- Validate product quantities
- Validate revenue calculations
- Ensure positive unit pricing
- Validate line-level profitability

Business metrics supported:
- Revenue
- Profit
- Basket analysis
- Discount analysis

Partition Strategy:
- order_year
- order_month

Output:
- silver_order_items

In [37]:
df_silver_order_items = (
    df_bronze_order_items
    .filter(col("quantity") > 0)
    .filter(col("line_total") >= 0)
    .filter(col("unit_price") > 0)
)

StatementMeta(, 6456375b-7f01-4a0d-b876-a18e7f14e1f4, 38, Finished, Available, Finished, False)

In [38]:
df_silver_order_items.write \
    .mode("overwrite") \
    .partitionBy("order_year", "order_month") \
    .format("delta") \
    .saveAsTable("silver_order_items")

StatementMeta(, 6456375b-7f01-4a0d-b876-a18e7f14e1f4, 39, Finished, Available, Finished, False)

## 6. Returns Data Transformation

This section transforms customer returns and refund transactions.

Transformation logic:
- Validate refund amounts
- Standardize return reasons
- Prepare return analytics datasets

Returns are modeled separately from sales transactions to preserve transactional history and support net revenue calculations.

Partition Strategy:
- return_year
- return_month

Output:
- silver_returns

In [39]:
df_silver_returns = (
    df_bronze_returns
    .filter(col("refund_amount") >= 0)
    .withColumn("return_reason", initcap(trim(col("return_reason"))))
)

StatementMeta(, 6456375b-7f01-4a0d-b876-a18e7f14e1f4, 40, Finished, Available, Finished, False)

In [40]:
df_silver_returns.write \
    .mode("overwrite") \
    .partitionBy("return_year", "return_month") \
    .format("delta") \
    .saveAsTable("silver_returns")

StatementMeta(, 6456375b-7f01-4a0d-b876-a18e7f14e1f4, 41, Finished, Available, Finished, False)

## 7. Inventory Snapshot Transformation

This section transforms warehouse inventory snapshot data.

Transformation logic:
- Validate stock quantities
- Validate units sold
- Standardize stock status labels
- Prepare inventory monitoring datasets

Partition Strategy:
- snapshot_year
- snapshot_month

Output:
- silver_inventory

In [41]:
df_silver_inventory = (
    df_bronze_inventory
    .filter(col("ending_stock") >= 0)
    .filter(col("units_sold") >= 0)
    .withColumn("stock_status", initcap(trim(col("stock_status"))))
)

StatementMeta(, 6456375b-7f01-4a0d-b876-a18e7f14e1f4, 42, Finished, Available, Finished, False)

In [42]:
df_silver_inventory.write \
    .mode("overwrite") \
    .partitionBy("snapshot_year", "snapshot_month") \
    .format("delta") \
    .saveAsTable("silver_inventory")

StatementMeta(, 6456375b-7f01-4a0d-b876-a18e7f14e1f4, 43, Finished, Available, Finished, False)

Location

In [43]:
df_silver_locations = (
    df_bronze_locations
    .dropDuplicates(["location_id"])
    .withColumn("city", initcap(trim(col("city"))))
    .withColumn("province", initcap(trim(col("province"))))
    .withColumn("country", initcap(trim(col("country"))))
    .withColumn("gta_region", trim(col("gta_region")))
)

StatementMeta(, 6456375b-7f01-4a0d-b876-a18e7f14e1f4, 44, Finished, Available, Finished, False)

In [44]:
df_silver_locations.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_locations")

StatementMeta(, 6456375b-7f01-4a0d-b876-a18e7f14e1f4, 45, Finished, Available, Finished, False)

## 8. Silver Layer Validation

This section validates:
- Row counts
- Partition generation
- Data quality checks
- Transactional consistency
- Inventory consistency

The Silver layer datasets are now ready for Gold layer dimensional modeling and reporting.

In [45]:
tables = [
    "silver_customers",
    "silver_products",
    "silver_orders",
    "silver_order_items",
    "silver_returns",
    "silver_inventory",
    "silver_locations"
]

for table in tables:
    count = spark.table(table).count()
    print(f"{table}: {count:,} rows")

StatementMeta(, 6456375b-7f01-4a0d-b876-a18e7f14e1f4, 46, Finished, Available, Finished, False)

silver_customers: 200 rows
silver_products: 50 rows
silver_orders: 20,252 rows
silver_order_items: 60,274 rows
silver_returns: 1,660 rows
silver_inventory: 600 rows
silver_locations: 10 rows


## Silver Layer Complete

The Silver layer transformation process has successfully:
- Cleansed raw retail datasets
- Standardized business data
- Applied validation rules
- Prepared datasets for enterprise analytics
